# 🕵️ Dia 66 — Comprometimento de e-mail corporativo (BEC) e detecção de domínios falsificados

**Investigador Cyber 90 Dias** · Fase 5 · Rank Jutsu `A`

> Baseado no *Day 66* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-5-fraud-social-engineering/day-66.md) (*Business email compromise and how spoofed domains are detected*), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Terça, 20 de outubro. O contas a pagar da Serra Azul recebeu três e-mails da "Gráfica Jatobá": um boleto reemitido, um pedido de chave Pix nova (**PIXFICT-L04**) e uma troca de banco no meio de uma conversa antiga.

Rafa: *"Começa pelo domínio, não pelo SPF."*

Neste notebook você vai:

1. **Ler `Authentication-Results` e `Received:`** com o módulo `email`.
2. Transformar o fluxograma do dia na função **`classificar_bec()`**.
3. Gerar **permutações de domínio** no estilo do dnstwist, só com stdlib.
4. Jogar o **quiz das seis classificações** e gerar `notas/dia66-cabecalhos-e-imitacoes.md`.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Todos os dados são **fictícios** (domínios `.example`, telefones `+55 00 00000-00NN`, chaves Pix `PIXFICT-*`).

## 1. Os três e-mails da Gráfica Jatobá

Os cabeçalhos estão embutidos (plano B offline). Repare que só o cabeçalho importa para a classificação técnica; o corpo importa para o checklist de sinais.

In [ ]:
# Domínio verdadeiro da fornecedora, visto em 2 anos de correspondência genuína.
DOMINIO_CONHECIDO = "graficajatoba.example"

# Três e-mails fictícios (mesmo conteúdo de caso-continuo/dados/dia-66/)
EMAILS = {
"BR-A": """Received: from mx.serraazul.example (mx.serraazul.example [192.0.2.10])
        by caixa.serraazul.example; Tue, 20 Oct 2026 09:02:11 -0300
Received: from smtp-out.envio-barato.example (unknown [203.0.113.55])
        by mx.serraazul.example; Tue, 20 Oct 2026 09:02:09 -0300
Authentication-Results: mx.serraazul.example;
  spf=fail (sender IP 203.0.113.55 not permitted by domain of graficajatoba.example)
  dkim=none;
  dmarc=fail (p=none) header.from=graficajatoba.example
From: "Financeiro Grafica Jatoba" <financeiro@graficajatoba.example>
Reply-To: financeiro.graficajatoba@webmail.example
Return-Path: <financeiro@graficajatoba.example>
Subject: URGENTE boleto atualizado - pagar hoje

Bom dia. O boleto da NF 4471 venceu e foi reemitido. Favor pagar o anexo ainda hoje,
antes das 14h, para nao suspendermos a entrega dos taloes. Estou em viagem, sem telefone.
""",
"BR-B": """Received: from mx.serraazul.example (mx.serraazul.example [192.0.2.10])
        by caixa.serraazul.example; Tue, 20 Oct 2026 10:15:40 -0300
Received: from mail.graficajatoba-cobranca.example (mail.graficajatoba-cobranca.example [198.51.100.61])
        by mx.serraazul.example; Tue, 20 Oct 2026 10:15:38 -0300
Authentication-Results: mx.serraazul.example;
  spf=pass (sender IP 198.51.100.61 permitted by domain of graficajatoba-cobranca.example)
  dkim=pass header.d=graficajatoba-cobranca.example;
  dmarc=pass (p=reject) header.from=graficajatoba-cobranca.example
From: "Financeiro Grafica Jatoba" <financeiro@graficajatoba-cobranca.example>
Reply-To: financeiro@graficajatoba-cobranca.example
Return-Path: <financeiro@graficajatoba-cobranca.example>
Subject: Atualizacao de dados bancarios - Grafica Jatoba

Prezados, por auditoria em nosso banco, os proximos pagamentos devem ser feitos
pela chave Pix PIXFICT-L04 (ficticia). Pedimos sigilo ate a conclusao da auditoria.
""",
"BR-C": """Received: from mx.serraazul.example (mx.serraazul.example [192.0.2.10])
        by caixa.serraazul.example; Wed, 21 Oct 2026 16:40:02 -0300
Received: from smtp.graficajatoba.example (smtp.graficajatoba.example [198.51.100.20])
        by mx.serraazul.example; Wed, 21 Oct 2026 16:40:00 -0300
Authentication-Results: mx.serraazul.example;
  spf=pass (sender IP 198.51.100.20 permitted by domain of graficajatoba.example)
  dkim=pass header.d=graficajatoba.example;
  dmarc=pass (p=quarantine) header.from=graficajatoba.example
From: "Financeiro Grafica Jatoba" <financeiro@graficajatoba.example>
Reply-To: financeiro@graficajatoba.example
Return-Path: <financeiro@graficajatoba.example>
Subject: Re: Re: NF 4471 e pedido de taloes de novembro

Ola, seguindo nossa conversa: mudamos de banco. O pagamento da NF 4471 deve ir
para a nova conta informada no PDF anexo. Obrigada!
""",
}
print(len(EMAILS), "e-mails carregados:", ", ".join(EMAILS))

## 2. Extrair o que o investigador lê

- **Domínio do From** (o que se compara primeiro);
- **spf / dkim / dmarc** e a **política** (`p=`) que aparece no `Authentication-Results`;
- o **`d=`** do DKIM (alinhado ou não com o From);
- o **primeiro `Received:` externo**: lendo de baixo para cima, o mais baixo é o mais perto do remetente.

In [ ]:
import re
from email import message_from_string
from email.utils import parseaddr

def dominio_de(endereco_bruto):
    """'Nome <a@b.example>' → 'b.example'."""
    _, end = parseaddr(endereco_bruto or "")
    return end.rsplit("@", 1)[-1].lower() if "@" in end else ""

def ler_cabecalho(texto):
    msg = message_from_string(texto)
    # Authentication-Results pode quebrar em várias linhas; o parser junta tudo.
    ar = " ".join((msg.get("Authentication-Results") or "").split())
    achar = lambda padrao: (re.search(padrao, ar, re.I) or [None, None])[1]   # 1º grupo ou None
    received = msg.get_all("Received") or []
    # O ÚLTIMO Received da lista é o mais baixo do cabeçalho = o mais perto do remetente.
    mais_baixo = " ".join(received[-1].split()) if received else ""
    ip = re.search(r"\[(\d+\.\d+\.\d+\.\d+)\]", mais_baixo)
    return {
        "from": dominio_de(msg["From"]),
        "reply_to": dominio_de(msg["Reply-To"]),
        "return_path": dominio_de(msg["Return-Path"]),
        "spf": achar(r"spf=(\w+)"),
        "dkim": achar(r"dkim=(\w+)"),
        "dkim_d": achar(r"header\.d=([\w.-]+)"),
        "dmarc": achar(r"dmarc=(\w+)"),
        "politica": (achar(r"\(p=(\w+)\)") or "sem registro").lower(),
        "primeiro_externo": ip.group(1) if ip else "?",
        "assunto": msg["Subject"],
        "corpo": msg.get_payload(),
    }

LIDOS = {nome: ler_cabecalho(txt) for nome, txt in EMAILS.items()}
for nome, c in LIDOS.items():
    print(f"\n{nome}  From={c['from']}  Reply-To={c['reply_to']}")
    print(f"   spf={c['spf']} dkim={c['dkim']} (d={c['dkim_d']}) dmarc={c['dmarc']} p={c['politica']}")
    print(f"   primeiro salto externo: {c['primeiro_externo']}")

## 3. O fluxograma vira função

Mesma ordem do original: **domínio primeiro**. Um domínio imitação passa em SPF, DKIM e DMARC, porque o golpista configurou tudo para o domínio dele.

In [ ]:
# Palavras que indicam pedido de mudança de pagamento, sigilo ou prazo (passo 3 do fluxograma)
GATILHOS_PEDIDO = ("conta", "banco", "pix", "boleto", "sigilo", "hoje", "urgente", "antes das")

def classificar_bec(dominio_from, dominio_conhecido, dmarc, dkim_d, texto_pedido):
    """Devolve (classificação, por quê) seguindo o fluxograma do Dia 66."""
    # Passo 1 — o domínio é EXATAMENTE o mesmo da correspondência genuína?
    if dominio_from != dominio_conhecido:
        return ("DOMÍNIO IMITAÇÃO",
                f"{dominio_from} ≠ {dominio_conhecido}; a autenticação pode passar, mas para o domínio errado")
    # Passo 2 — domínio idêntico: agora sim a autenticação diz algo.
    alinhado = (dmarc == "pass") and (dkim_d in (None, dominio_from))
    if not alinhado:
        return ("REMETENTE FALSIFICADO",
                f"domínio verdadeiro, mas dmarc={dmarc}: servidor não autorizado pelo domínio")
    # Passo 3 — tudo passa: o único sinal que sobra é o PEDIDO.
    if any(g in texto_pedido.lower() for g in GATILHOS_PEDIDO):
        return ("POSSÍVEL CAIXA COMPROMETIDA",
                "todas as checagens passam para o domínio certo; o pedido muda pagamento/impõe prazo")
    return ("PROVAVELMENTE GENUÍNO", "domínio idêntico, autenticação alinhada, pedido comum")

for nome, c in LIDOS.items():
    classe, motivo = classificar_bec(c["from"], DOMINIO_CONHECIDO, c["dmarc"], c["dkim_d"],
                                     c["assunto"] + " " + c["corpo"])
    print(f"{nome}: {classe}\n   ↳ {motivo}")
# Em todos os três casos a saída é a mesma: verificar por fora do e-mail,
# ligando para um número JÁ CADASTRADO, nunca para um que esteja na mensagem.

> 💡 **Por que BR-A chegou à caixa de entrada?** `dmarc=fail`, mas a política da gráfica é `p=none`: ela publicou DMARC só para observar, sem mandar recusar. A falha é de configuração do **domínio falsificado**, não do servidor da cooperativa. Recomendação para a Marta passar à gráfica: evoluir para `p=quarantine` e depois `p=reject`.

## 4. Permutações de domínio (o dnstwist em miniatura)

O original usa `dnstwist --registered`. Aqui geramos as permutações com stdlib para entender a lógica; sem rede, não dá para saber quais estão registradas, então cruzamos com uma lista **fictícia** de "domínios novos do dia".

In [ ]:
def permutacoes(dominio):
    """Gera candidatas de imitação, cada uma com o tipo (nomes iguais aos do dnstwist)."""
    nome, tld = dominio.rsplit(".", 1)
    cand = {}
    for i in range(len(nome)):
        cand[nome[:i] + nome[i+1:]] = "omission"                    # tira uma letra
        cand[nome[:i] + nome[i] * 2 + nome[i+1:]] = "repetition"    # dobra uma letra
        if 0 < i < len(nome):
            cand[nome[:i] + "-" + nome[i:]] = "hyphenation"         # hífen no meio
        if i < len(nome) - 1:                                        # troca duas vizinhas
            cand[nome[:i] + nome[i+1] + nome[i] + nome[i+2:]] = "transposition"
    for a, b in {"o": "0", "l": "1", "i": "l", "m": "rn", "a": "4"}.items():
        if a in nome:
            cand[nome.replace(a, b, 1)] = "homoglyph"                # letra parecida
    for extra in ("cobranca", "financeiro", "boleto", "pagamentos", "seguranca", "pix"):
        cand[f"{nome}-{extra}"] = "addition"                         # palavra a mais
    cand.pop(nome, None)                                             # o original não é imitação
    resultado = {f"{n}.{tld}": t for n, t in cand.items() if n}
    for outro_tld in ("example.com", "example.net"):                 # troca de TLD (fictícia)
        resultado[f"{nome}.{outro_tld}"] = "tld-swap"
    return resultado

perm_grafica = permutacoes(DOMINIO_CONHECIDO)
perm_coop = permutacoes("serraazul.example")
print(len(perm_grafica), "candidatas para a gráfica;", len(perm_coop), "para a cooperativa")

# "Feed" fictício de domínios registrados nos últimos dias (no mundo real: novos registros / CT logs)
NOVOS_DO_DIA = ["graficajatoba-cobranca.example", "serraazul-seguranca.example",
                "coopserra-azul.example", "padaria-boa.example", "serraazul-pix.example"]

print("\nInterseção com o feed de domínios novos:")
for d in NOVOS_DO_DIA:
    tipo = perm_grafica.get(d) or perm_coop.get(d)
    if tipo:
        print(f"  ⚠️  {d:35} tipo={tipo}")
    elif any(p in d for p in ("serraazul", "serra-azul", "graficajatoba")):
        # coopserra-azul não sai das permutações simples: por isso o dnstwist tem dicionários
        print(f"  🔎 {d:35} contém o nome, mas fora das regras simples: olho humano")

## 5. 🎮 Quiz das seis classificações

Três casos do original (Harborline) e três do caso brasileiro. Responda **1** (remetente falsificado), **2** (domínio imitação) ou **3** (caixa comprometida).

`MODO_JOGO = False` para só ver o gabarito.

In [ ]:
MODO_JOGO = True

QUIZ = [
    ("🇺🇸 A", "From ceo@harborline.example; spf=fail, dkim=none, dmarc=fail (p=none); pede cartões-presente antes das 14h", "1",
     "Domínio verdadeiro, servidor não autorizado. p=none deixou chegar."),
    ("🇺🇸 B", "From billing@harborIine.example (I maiúsculo); tudo pass, d=harborIine.example; domínio de 9 dias", "2",
     "A autenticação passou para o domínio ERRADO."),
    ("🇺🇸 C", "From billing@harborline.example; tudo pass, d=harborline.example; dentro de conversa de 3 semanas; login estranho confirmado", "3",
     "Tudo passa para o domínio certo; só a ligação ao número cadastrado pegaria."),
]
# Os três brasileiros vêm da função, para o quiz e o código concordarem sempre.
MAPA = {"REMETENTE FALSIFICADO": "1", "DOMÍNIO IMITAÇÃO": "2", "POSSÍVEL CAIXA COMPROMETIDA": "3"}
for nome, c in LIDOS.items():
    classe, motivo = classificar_bec(c["from"], DOMINIO_CONHECIDO, c["dmarc"], c["dkim_d"], c["assunto"] + " " + c["corpo"])
    QUIZ.append((f"🇧🇷 {nome}", f"From @{c['from']}; spf={c['spf']} dkim={c['dkim']} dmarc={c['dmarc']} (p={c['politica']}); \"{c['assunto']}\"",
                 MAPA[classe], motivo))

NOMES = {"1": "remetente falsificado", "2": "domínio imitação", "3": "caixa comprometida"}
acertos = 0
for rotulo, descricao, certo, porque in QUIZ:
    print(f"\n[{rotulo}] {descricao}")
    if MODO_JOGO:
        r = ""
        while r not in ("1", "2", "3"):
            r = input("   1 falsificado / 2 imitação / 3 caixa comprometida: ").strip()
        acertos += (r == certo)
        print("   ✅" if r == certo else "   ❌")
    print(f"   Gabarito: {NOMES[certo]} — {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}  (Checkpoint BR: pelo menos 5)")

## 6. 📝 O artefato do dia

In [ ]:
#@title Para quem a Marta liga { display-mode: "form" }
ligacao_BR_A = "Banco/MED se pagou; gráfica pelo número cadastrado para avisar da falsificação e do p=none"  #@param {type:"string"}
ligacao_BR_B = "Banco/MED se pagou; registro do domínio graficajatoba-cobranca para derrubada; polícia"  #@param {type:"string"}
ligacao_BR_C = "Gráfica pelo número cadastrado (o incidente é dela); banco/MED se pagou; jurídico (LGPD art. 48)"  #@param {type:"string"}

from pathlib import Path
ligacoes = {"BR-A": ligacao_BR_A, "BR-B": ligacao_BR_B, "BR-C": ligacao_BR_C}
d = lambda s: (s or "").replace(".", "[.]")     # desarma domínios no relatório
linhas = ["# Cabeçalhos e domínios imitação — Dia 66 (Operação Isca, fictício)", "",
          "| E-mail | Domínio From | spf | dkim (d=) | dmarc (p=) | 1º salto externo | Classificação | Ligação |",
          "|---|---|---|---|---|---|---|---|"]
for nome, c in LIDOS.items():
    classe, _ = classificar_bec(c["from"], DOMINIO_CONHECIDO, c["dmarc"], c["dkim_d"], c["assunto"] + " " + c["corpo"])
    linhas.append(f"| {nome} | {d(c['from'])} | {c['spf']} | {c['dkim']} ({d(c['dkim_d'])}) | {c['dmarc']} ({c['politica']}) "
                  f"| {c['primeiro_externo']} | {classe} | {ligacoes[nome]} |")
linhas += ["", "## Domínios imitação no feed do dia", "", "| Domínio | Tipo |", "|---|---|"]
for dom in NOVOS_DO_DIA:
    tipo = perm_grafica.get(dom) or perm_coop.get(dom)
    if tipo:
        linhas.append(f"| {d(dom)} | {tipo} |")
linhas += ["", "Indicador para o Dia 68: chave PIXFICT-L04 (fictícia), informada no BR-B.", ""]

Path("notas").mkdir(exist_ok=True)
saida = Path("notas/dia66-cabecalhos-e-imitacoes.md")
saida.write_text("\n".join(linhas), encoding="utf-8")
print("\n".join(linhas))
try:
    from google.colab import files
    files.download(str(saida))
except ImportError:
    print("(Fora do Colab: o arquivo ficou em notas/)")

### 🧠 Camada 8

O BR-C não tem nada de errado no cabeçalho. Levine chama de **padrão de verdade** a nossa presunção de que o outro diz a verdade; ela só cai com um gatilho. O gatilho que funciona é de processo: *mudança de pagamento por e-mail sempre dispara uma ligação para o número já cadastrado.* Recomende o processo, não "mais atenção".

## ✅ Antes de fechar o Dia 66

- [ ] Quiz com pelo menos 5/6
- [ ] `notas/dia66-cabecalhos-e-imitacoes.md` gerado
- [ ] Você explica por que BR-B passou no DMARC e mesmo assim é golpe
- [ ] PIXFICT-L04 anotada como indicador para o Dia 68

**Amanhã (Dia 67):** a dona da conta L-03 respondeu a um anúncio de emprego. Falso emprego, recrutamento de laranja e um checklist em que cada célula precisa de citação.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>